# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook provides a walkthrough for loading, exploring, and processing the **FAIR^2 Clinical Genotype-Phenotype Dataset** using the `mlcroissant` library. All dataset entities—including record sets, fields, and columns—are referenced by their `@id` as per the Croissant schema syntax.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")
print(f"Date Published: {metadata.datePublished}")
print(f"Dataset @id: {metadata['@id']}")

## 2. Data Overview
Review available record sets, their fields, columns, and their `@id`s.

`mlcroissant` allows you to explore the record sets as defined by the Croissant schema, which are entry points to the structured tables in the dataset.

In [ ]:
# Examine available record sets in the dataset
record_sets = dataset.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"- Name: {rs.name}\n  @id: {rs['@id']}\n  Description: {getattr(rs, 'description', '<No description>')}\n")

# Display info for each record set
for rs in record_sets:
    print(f"Record set: {rs.name} (@id: {rs['@id']})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Name: {field.name}\n      @id: {field['@id']}\n      DataType: {getattr(field, 'dataType', '')}")
    print("  Columns:")
    for column in rs.columns:
        print(f"    - Name: {column.name}\n      @id: {column['@id']}\n      Source: {getattr(column, 'source', '')}")
    print()

## 3. Data Extraction
Load tabular data from each record set into pandas DataFrames for analysis.

Refer to the record sets' `@id`s as shown above.

For demonstration, we load all available record sets.

In [ ]:
# Collect all record set @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for record set @id: {rs_id}. Reason: {e}")

# Display columns from each loaded DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns for record set @id: {rs_id}\n{df.columns.tolist()}\n")

# For demonstration, select the first loaded record set
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping.

Let's choose a numeric field from the first record set for demonstration.

In [ ]:
# EDA on the first record set
import numpy as np

if len(dataframes) > 0:
    df = dataframes[first_rs_id]
    print(f"Columns in record set @id {first_rs_id}: {df.columns.tolist()}")
    
    # Try to automatically select a numeric field
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    # If not found, fallback
    if numeric_field is None:
        possible_numeric = [c for c in df.columns if 'age' in c.lower()
                         or 'level' in c.lower()
                         or 'height' in c.lower()
                         or 'frequency' in c.lower()]
        if possible_numeric:
            numeric_field = possible_numeric[0]

    if numeric_field is not None:
        print(f"Using numeric field: {numeric_field}")
        # Choose threshold for filtering
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
        filtered_df = df[df[numeric_field] > threshold]

        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for demonstration.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show a histogram for the numeric field and a boxplot (if feasible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and numeric_field is not None:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in record set @id {first_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    # Boxplot by group field
    if group_field:
        plt.figure(figsize=(6, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load, inspect, extract, and analyze a clinical dataset using `mlcroissant` by referencing all structural entities using their `@id`. Use the Croissant schema and field `@id`s as canonical pointers for reproducible downstream data processing.

**Key findings:**
- Dataset metadata and table structure are directly accessible from the Croissant schema.
- All code examples reference record sets, fields, and columns by their `@id`, ensuring schema consistency.
- The dataset is designed for rare disease research and genotype-phenotype correlation, but its small sample size may limit generalization and modeling -- see documented biases and use cases.

Feel free to extend this notebook to integrate more advanced analyses, visualizations, or downstream ML pipelines.